# Automatic Differentiation

A personal reference for how gradients are actually computed in modern ML frameworks. Every worked example is verified with code. Only `numpy` is needed.

**Contents**
1. Three ways to differentiate — and why AD
2. Forward mode: dual numbers
3. Reverse mode: the tape
4. Backprop by hand on a toy ODE (2 Euler steps)
5. Forward vs reverse: the parameter count argument
6. What PyTorch does under the hood
7. Quick self-test

## 1. Three ways to differentiate — and why AD

| Method | Idea | Problem |
|---|---|---|
| **Symbolic** | manipulate the expression tree algebraically | expression swell; doesn't compose with control flow |
| **Numerical** (finite diff) | (f(x+ε) − f(x)) / ε | O(n) evaluations for n params; roundoff error |
| **Automatic (AD)** | apply chain rule to the *program*, not the expression | exact (to floating point); O(1) passes for all gradients |

AD is neither symbolic nor numerical. It executes the program as usual but carries extra bookkeeping (either extra numbers or a tape of operations) that lets the chain rule run automatically.

**Key fact:** AD produces the exact mathematical derivative of whatever the program actually computed, to floating-point precision. It does not approximate.

## 2. Forward mode: dual numbers

Attach a "tangent" ẋ to every number x. Run the program replacing every real operation with its dual-number counterpart:

| Operation | Primal | Tangent (chain rule) |
|---|---|---|
| z = x + y | z = x + y | ż = ẋ + ẏ |
| z = x · y | z = x · y | ż = ẋ·y + x·ẏ |
| z = sin(x) | z = sin(x) | ż = cos(x) · ẋ |
| z = exp(x) | z = exp(x) | ż = exp(x) · ẋ |

To get dL/dw₁: seed ẇ₁ = 1, ẇ₂ = 0, …, run once → ẋ output = dL/dw₁.  
To get dL/dw₂: seed ẇ₂ = 1, rest 0, run again.

**Cost:** 1 forward pass per input you want the derivative *with respect to*. For n parameters → n passes.

**Good when:** many outputs, few inputs (e.g. a Jacobian of a 1→1000 map, or 2 parameters).

In [ ]:
import numpy as np

# Forward-mode AD via dual numbers for f(x, w) = exp(w * x)
# We want df/dw at x=1, w=0.5

class Dual:
    """A number plus its tangent: (value, derivative)."""
    def __init__(self, val, dot=0.0):
        self.val = val
        self.dot = dot
    def __mul__(self, other):
        return Dual(self.val * other.val,
                    self.dot * other.val + self.val * other.dot)  # product rule
    def __repr__(self):
        return f"Dual(val={self.val:.4f}, dot={self.dot:.4f})"

def dual_exp(d):
    ev = np.exp(d.val)
    return Dual(ev, ev * d.dot)   # chain rule: d/d? exp(u) = exp(u) * u'

x = Dual(1.0, 0.0)   # x is a constant w.r.t. w, so tangent = 0
w = Dual(0.5, 1.0)   # seed: we want derivative w.r.t. w, so tangent = 1

result = dual_exp(w * x)   # computes exp(w*x) and d/dw exp(w*x)
print(result)
print(f"Exact df/dw = x * exp(w*x) = {1.0 * np.exp(0.5):.4f}")

## 3. Reverse mode: the tape

Run the program forward, **recording every operation** (the "tape" or "Wengert list"). Then walk the tape backward, propagating the output gradient back to each input via the chain rule.

Each node v in the computation graph accumulates a **bar** v̄ = ∂L/∂v. Backward rules ("vjp" = vector-Jacobian product):

| Forward op | Backward rule |
|---|---|
| z = x + y | x̄ += z̄ ; ȳ += z̄ |
| z = x · y | x̄ += z̄ · y ; ȳ += z̄ · x |
| z = exp(x) | x̄ += z̄ · exp(x) |
| z = sin(x) | x̄ += z̄ · cos(x) |

**Cost:** 1 forward pass (to record) + 1 backward pass → gradient w.r.t. **all** inputs at once.

**Good when:** few outputs (e.g. one scalar loss), many inputs (millions of parameters). This is why every ML framework defaults to reverse mode.

**Memory cost:** the tape stores all intermediate values from the forward pass. Deeper networks = more memory. The adjoint method (see [adjoint_sensitivity.ipynb](adjoint_sensitivity.ipynb)) is a way to avoid this cost for ODE models.

In [ ]:
# Minimal reverse-mode AD for f(x, w) = exp(w * x)
# Forward pass records operations; backward pass replays them in reverse.

class Node:
    def __init__(self, val, backward_fn=None, deps=()):
        self.val = val
        self.bar = 0.0          # will hold d(output)/d(this node)
        self._backward = backward_fn
        self._deps = deps       # nodes this one depends on

    def backward(self):
        if self._backward:
            self._backward(self.bar)

def mul(a, b):
    z = Node(a.val * b.val)
    def bwd(g):
        a.bar += g * b.val   # dz/da = b
        b.bar += g * a.val   # dz/db = a
    z._backward = bwd
    return z

def exp_node(a):
    ev = np.exp(a.val)
    z = Node(ev)
    def bwd(g):
        a.bar += g * ev      # d exp(a) / da = exp(a)
    z._backward = bwd
    return z

x = Node(1.0)
w = Node(0.5)

# Forward pass
p = mul(w, x)
out = exp_node(p)       # out = exp(w*x)

# Backward pass
out.bar = 1.0           # dL/dout = 1 (out is the loss)
out.backward()          # propagate to p
p.backward()            # propagate to w and x

print(f"dL/dw = {w.bar:.4f}   exact = {1.0 * np.exp(0.5):.4f}")
print(f"dL/dx = {x.bar:.4f}   exact = {0.5 * np.exp(0.5):.4f}")

## 4. Backprop by hand on a toy ODE (2 Euler steps)

Model: **dx/dt = w·x**, Euler with h = 0.5, x₀ = 1, w = 0.5, loss L = x₂.

x_{k+1} = x_k + h·w·x_k = (1 + hw)·x_k, and 1 + hw = 1.25.

**Forward:** x₀ = 1 → x₁ = 1.25 → x₂ = 1.5625 = L

**Backward** — walk the tape in reverse (λ_k = ∂L/∂x_k is the bar of each node):
- λ₂ = ∂L/∂x₂ = 1
- x₂ = (1 + hw)·x₁ → λ₁ = λ₂·(1 + hw) = 1.25; grad contribution = λ₂·h·x₁ = 0.625
- x₁ = (1 + hw)·x₀ → λ₀ = λ₁·(1 + hw) = 1.5625; grad contribution = λ₁·h·x₀ = 0.625

**dL/dw = 0.625 + 0.625 = 1.25.** Closed form: L = (1 + hw)², dL/dw = 2h(1 + hw) = 1.25 ✓

The tape stored x₁ and x₀ — both needed during the backward pass.

> **This is exactly reverse-mode AD applied to a loop.** The "tape" is the list [x₀, x₁, x₂]. The backward rules are just the product-rule entries for the Euler step.

In [ ]:
import numpy as np

w, h, x0 = 0.5, 0.5, 1.0

# Forward pass: record every intermediate state (the "tape")
xs = [x0]
for _ in range(2):
    xs.append(xs[-1] + h * w * xs[-1])   # x_{k+1} = (1 + hw) * x_k
print("tape (stored states):", xs)

# Backward pass: walk tape in reverse
lam = 1.0   # dL/dx2 = 1
grad = 0.0
for k in [1, 0]:                          # step 2 backward, then step 1 backward
    grad += lam * h * xs[k]               # contribution of this step to dL/dw
    lam  *= (1 + h * w)                   # pass gradient back to x_k

print(f"dL/dw (backprop) = {grad:.4f}")
print(f"closed form      = {2 * (1 + h*w) * h:.4f}")

## 5. Forward vs reverse: the parameter count argument

Suppose L: ℝⁿ → ℝ (n parameters, one scalar loss, the standard ML setup).

| Mode | Passes needed | Memory during pass | Best for |
|---|---|---|---|
| **Forward** | n (one per parameter) | O(1) — no tape | n is small (e.g. 2 params, or computing a full Jacobian of a small output) |
| **Reverse** | 1 forward + 1 backward | O(depth) — stores tape | n is large (millions of weights), one scalar loss |

For n = 10⁸ parameters and one loss, forward mode would take 10⁸ passes. Reverse mode takes 2 passes (one forward, one backward) and returns all 10⁸ gradients. The price is storing the tape.

**The adjoint method** (see [adjoint_sensitivity.ipynb](adjoint_sensitivity.ipynb)) is a way to use reverse-mode AD on ODE models while paying only O(1) memory — by recomputing intermediate states instead of storing them.

In [ ]:
# Illustrate cost: forward mode needs one pass per parameter
# Reverse mode needs one backward pass for all parameters

def f(params):
    """Toy loss: sum of exp(p_i * x_i) for fixed x."""
    x = np.arange(1, len(params) + 1, dtype=float)
    return np.sum(np.exp(params * x))

params = np.array([0.1, 0.2, 0.3, 0.4])
x = np.arange(1, 5, dtype=float)

# Numerical finite differences (proxy for "n forward passes")
eps = 1e-5
grad_fd = np.array([(f(params + eps * np.eye(len(params))[i]) - f(params)) / eps
                    for i in range(len(params))])

# Exact (closed form): df/dp_i = x_i * exp(p_i * x_i)
grad_exact = x * np.exp(params * x)

print("finite diff (4 passes):", np.round(grad_fd, 4))
print("exact               :", np.round(grad_exact, 4))
print("\nFor n=1e8 parameters, finite diff needs 1e8 passes; reverse AD needs 1.")

## 6. What PyTorch does under the hood

PyTorch uses **dynamic reverse-mode AD** ("define-by-run" or eager mode):

1. Every `Tensor` with `requires_grad=True` records operations as they happen, building a DAG.
2. `.backward()` walks the DAG in reverse, calling the registered VJP ("grad_fn") for each node.
3. Gradients accumulate in `.grad`.

The graph is built and discarded per forward pass (unless `retain_graph=True`), so loops, conditionals, and dynamic shapes work transparently.

**Key vocabulary:**
- `grad_fn` — the backward function stored on each tensor (e.g. `MmBackward`, `ExpBackward`)
- `retain_graph=True` — keep the tape after `.backward()` (needed for multiple backward calls)
- `torch.no_grad()` — skip recording; saves memory and compute when gradients aren't needed
- `torch.utils.checkpoint` — gradient checkpointing: re-run some forward segments during backward to trade compute for memory (O(√n) memory instead of O(n))

**What PyTorch cannot do for general networks:** the adjoint memory trick (recomputing states by running the network backward), because most layers (ReLU, max-pool, size-changing) lose information. For ODE models, `torchdiffeq.odeint_adjoint` adds a custom `grad_fn` that does exactly this — but still calls PyTorch's AD for each VJP inside it.

In [ ]:
try:
    import torch

    # Same toy ODE loss via PyTorch autograd
    w = torch.tensor(0.5, requires_grad=True)
    h = 0.5
    x = torch.tensor(1.0)

    # Forward pass — PyTorch records every operation
    for _ in range(2):
        x = x + h * w * x   # x is a new tensor each step; grad_fn is built

    x.backward()  # reverse pass
    print(f"dL/dw via PyTorch autograd = {w.grad.item():.4f}   (expected 1.25)")
    print(f"grad_fn on final x: {x.grad_fn}")

except ImportError:
    print("torch not installed — snippet skipped (pip install torch)")

## 7. Quick self-test

<details><summary>What does AD compute, exactly?</summary>The exact derivative of whatever the program computed, to floating-point precision — not an approximation.</details>

<details><summary>What is a dual number?</summary>A pair (value, tangent) where arithmetic follows the chain rule. One forward pass with seed ẇ=1 gives dL/dw.</details>

<details><summary>Why does reverse mode dominate in ML?</summary>n parameters, 1 loss → reverse mode costs 1 forward + 1 backward pass for all gradients. Forward mode would need n passes.</details>

<details><summary>What is the tape?</summary>The record of all intermediate values during the forward pass. The backward pass walks it in reverse to apply chain rule VJPs.</details>

<details><summary>What does gradient checkpointing do?</summary>Instead of storing all activations, it re-runs segments of the forward pass during backward. Reduces memory from O(n) to O(√n) at the cost of ~1 extra forward pass.</details>

<details><summary>Is backprop the same as reverse-mode AD?</summary>Yes. Backprop is reverse-mode AD applied to the composition of layer operations.</details>

<details><summary>What's the link to adjoint sensitivity?</summary>The adjoint method is a way to do reverse-mode AD on ODE solves without keeping the full tape — it recomputes intermediate states by running the ODE backward. See adjoint_sensitivity.ipynb.</details>